# EZStats Pipeline v3

Run setup cells 1-5, then run each match separately. Messi is first. Each match saves its video, JSON files, and full log to Drive before its cell completes. This is a candidate pipeline to validate, not a measured accuracy guarantee.


### 1. GPU

Select Runtime > Change runtime type > T4 GPU before running setup.


In [ ]:
import torch
print(torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))


### 2. Drive


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/ezstats')
assert DRIVE.is_dir()


### 3. Code

Local edits must be committed and pushed before this cell can fetch them. Pulling code does not update an already-open notebook; open this v3 notebook separately.


In [ ]:
import subprocess
from pathlib import Path

def run_checked(command, cwd=None):
    with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

REPO = Path('/content/EZStatsAIWorker')
if REPO.exists():
    run_checked(['git', 'checkout', 'test'], cwd=REPO)
    run_checked(['git', 'pull', '--ff-only'], cwd=REPO)
else:
    run_checked(['git', 'clone', '-b', 'test', 'https://github.com/MattyKKS/EZStatsAIWorker.git', str(REPO)])
run_checked(['git', 'log', '-1', '--oneline'], cwd=REPO)
assert (REPO / 'run_pipeline_v3.py').exists(), 'Pull the v3 changes first.'


### 4. Dependencies


In [ ]:
import os, sys
run_checked([sys.executable, '-m', 'pip', 'install', '-e', str(REPO) + '[ml,appearance]', 'ultralytics==8.4.48', 'supervision==0.25.1', 'lap>=0.5.12', 'git+https://github.com/roboflow/sports.git@42c80c06b6b65a7f89455b89fe31cdf4c38ba227'])
os.environ['PYTHONPATH'] = str(REPO / 'src')
run_checked([sys.executable, '-c', "import ez_worker, ultralytics, supervision, sports, torch; print(ultralytics.__version__, supervision.__version__); assert torch.cuda.is_available(), 'CUDA unavailable in pipeline interpreter'; print('Pipeline GPU:', torch.cuda.get_device_name(0))"], cwd=REPO)


### 5. Models and videos

Models are copied to runtime storage. The run helper also copies each selected video locally before processing it, avoiding repeated reads through Drive.


In [ ]:
import shutil
for name in ('artifacts', 'data'):
    assert (DRIVE / name).is_dir(), str(DRIVE / name)
models = REPO / 'artifacts'
if models.is_symlink():
    models.unlink()
required_models = [
    'training/player_detector_v2/weights/best.pt',
    'ball/football-ball-detection.pt',
    'pitch/football-pitch-detectionV2.pt',
]
for relative in required_models:
    source, destination = DRIVE / 'artifacts' / relative, models / relative
    assert source.is_file(), f'Missing model: {source}'
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f'Copying model: {relative} ({source.stat().st_size / 1e6:.0f} MB)', flush=True)
    shutil.copy2(source, destination)
    print('  Ready', flush=True)
video_link = REPO / 'data'
if not video_link.exists() and not video_link.is_symlink():
    video_link.symlink_to(DRIVE / 'data', target_is_directory=True)
assert (video_link / 'raw' / 'leo_messi_30pass.mp4').is_file(), 'Messi video missing under data/raw'
print('Videos:', [p.name for p in (video_link / 'raw').glob('*.mp4')])
print('Setup complete. Run 6A for Messi.')


### 6A. Messi


In [ ]:
import runpy
from pathlib import Path
REPO = Path('/content/EZStatsAIWorker')
# Run the logging helper in this kernel so its prints appear in the cell.
runpy.run_path(str(REPO / 'scripts/run_colab_clip.py'))['main'](['leo_messi_30pass.mp4'])


### 6B. Original benchmark


In [ ]:
import runpy
from pathlib import Path
REPO = Path('/content/EZStatsAIWorker')
# Run the logging helper in this kernel so its prints appear in the cell.
runpy.run_path(str(REPO / 'scripts/run_colab_clip.py'))['main'](['08fd33_4.mp4'])


### 6C. Brighton


In [ ]:
import runpy
from pathlib import Path
REPO = Path('/content/EZStatsAIWorker')
# Run the logging helper in this kernel so its prints appear in the cell.
runpy.run_path(str(REPO / 'scripts/run_colab_clip.py'))['main'](['BrightonGoal.mp4'])


### 6D. Mason Mount


In [ ]:
import runpy
from pathlib import Path
REPO = Path('/content/EZStatsAIWorker')
# Run the logging helper in this kernel so its prints appear in the cell.
runpy.run_path(str(REPO / 'scripts/run_colab_clip.py'))['main'](['19PassesAndMasonGoal.mp4'])


### Resume an interrupted finalization

The printed run directory contains raw tracks and crops after detection finishes. To redo team assignment, events, and rendering without rerunning detection, use `python run_pipeline_v3.py --resume outputs/<run>`. Results from a Colab runtime still need to be copied to Drive. Retain the original run log and `run_manifest.json` when comparing results.
